# Semenov, Kim, Silkuniene & Pakhomov (2025): sine waves and temporal interference stimulation

Paper. Semenov I, Kim V, Silkuniene G, Pakhomov AG (2025). Excitation and polarization of isolated
neurons by high-frequency sine waves for temporal interference stimulation. *Cell Reports Physical
Science* 6, 102660. doi:10.1016/j.xcrp.2025.102660. The claims file was written from the NIH author
manuscript (PMC12320893) with its STAR Methods and supplement.

Dataset. DANDI:001274, version 0.250805.1401 (the most recent published version). The paper names
no version. The dandiset has two published versions, 0.250701.1622 and 0.250805.1401, with the same 232
assets and the same number of bytes; the later one is pinned as the protocol asks (author, step 1.3), and
the notebook checks below that the asset lists are the same. DANDI:001274 holds the strobe images of
Figures 5, 6 and S3. The claims file assigns the other figures to other dandisets of the paper, read at
their most recent published versions, for the same reason: 001030 (0.240520.1924), 001032
(0.240527.1432), 001033 (0.240527.1432), 001035 (0.240527.1432), 001271 (0.250805.1352), 001272
(0.250805.1358) and 001273 (0.250805.1356). 001034 (0.240527.1431) is checked and found to be a second
deposit of 001032.

Claims (claims/001274/10.1016-j.xcrp.2025.102660.yaml):

| Claim | Data | Here |
|---|---|---|
| fig1-pure-sine-excites-all | 001032, 001033, 001035 | tested |
| fig1-onset-delay | same | tested |
| fig1-max-rate-ceiling | same | tested |
| fig1-excitation-block | same | tested |
| fig2d-20khz-twofold-threshold | same | tested |
| fig2d-modulated-thresholds-higher | same | tested |
| threshold-2khz-unmodulated | same | tested |
| fig2-modulated-locks-to-beats | same | tested |
| fig2c-pure-more-aps | same | tested |
| fig3b-low-conductivity-halves-threshold | none labelled | blocked: no 250-ms session is labelled as sodium gluconate |
| fig3c-sustained-firing-2s | 001030 | tested |
| figs1-di8-anepps-control | 001272 | tested |
| figs2-ttx-blocks-depolarization | 001271 | tested |
| fig4-orientation-time-constants | 001273 | blocked: no ROIs, no cell shape or orientation groups |
| fig4-no-pole-difference | 001273 | blocked: no ROIs |
| fig4-discharging-slower | 001273 | blocked: no ROIs, no groups |
| fig4-pooled-time-constants | 001273 | blocked: no ROIs |
| fig5-carrier-antiphasic, fig5-whole-cell-depolarization, fig5-sine-asymmetry-c7, figs3-sine-asymmetry-c8, fig5fg-no-migration | 001274 | blocked: no ROIs, cells C7 and C8 cannot be identified |
| fig6b-pulse-sensitivity, fig6c-sine-sensitivity, fig6d-sine-less-than-pulse | 001274 | blocked: no ROIs; rebuilt ROIs shown as a variant |
| threshold-depolarization-estimate | 001274 and 001032 to 001035 | blocked: depends on fig6d |

Method details the claims file leaves open are taken from the paper and cited in `ti_sine.py`. Three
choices are assumptions: the soma ROI (the whole 16 x 16 frame, or its brighter half, both run), the field
(36 V/cm times half the peak-to-peak voltage, or times the whole of it, both run), and, in this second version,
two conditions added to AP reading (a), a 5% prominence and a 1-ms half-width, which keep noise on a
depolarized plateau from counting as APs (tested in section 3).

In [ ]:
import json, re, time, collections
import numpy as np
import pandas as pd
import h5py, remfile
import matplotlib.pyplot as plt
from scipy import stats
import ti_sine as T

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
t_start = time.time()
print({d: v for d, v in T.VERSIONS.items()})

## 1. What the dandisets hold

Asset lists and session descriptions for every dandiset the claims use, read from the DANDI API at the
pinned versions.

In [ ]:
assets = {d: T.list_assets(d) for d in T.VERSIONS}
for d, a in assets.items():
    subjects = {x["path"].split("/")[0] for x in a}
    print(f"{d} {T.VERSIONS[d]}: {len(a)} assets, {len(subjects)} subjects, "
          f"{sum(x['size'] for x in a) / 1e9:.2f} GB")
# the two published versions of DANDI:001274
older = T._get(f"{T.API}/dandisets/001274/versions/0.250701.1622/assets/?page_size=1000")
older_paths = {(x["path"], x["size"]) for x in older["results"]}
newer_paths = {(x["path"], x["size"]) for x in assets["001274"]}
print("001274 versions 0.250701.1622 and 0.250805.1401 have the same paths and sizes:",
      older_paths == newer_paths, f"({len(older_paths)} and {len(newer_paths)} assets)")

In [ ]:
# One strobe file of DANDI:001274, everything it holds (the long schema strings are left out).
def listing(url):
    out = []
    with h5py.File(remfile.File(url), "r") as f:
        def visit(name, obj):
            if name.startswith("specifications"):
                return
            if isinstance(obj, h5py.Dataset):
                v = obj[()] if obj.size < 4 else None
                out.append(f"D {name} {obj.shape} {obj.dtype} {'' if v is None else str(v)[:120]}")
            else:
                out.append(f"G {name} {dict(obj.attrs).get('neurodata_type', '')}")
        f.visititems(visit)
    return out
print("\n".join(listing(assets["001274"][0]["url"])))
print()
print("\n".join(listing(assets["001032"][0]["url"])))

### DANDI:001274: sessions per subject

The stimulus, voltage and tetrodotoxin are read from each session's description. Frame counts and
session start times are read from the files' headers, without the images.

In [ ]:
def strobe_kind(desc):
    k = ("pulse" if "rectangular pulse" in desc else "sine 100 Hz" if "100Hz" in desc
         else "sine 154 Hz" if "154Hz" in desc else "sine")
    v = float(re.search(r"at ([\d,]+)V", desc).group(1).replace(",", "."))
    return k, v, "TTX" if "presence of the sodium" in desc else "vehicle"

def strobe_header(a):
    with h5py.File(remfile.File(a["url"]), "r") as f:
        return f["acquisition/StrobeImaging/data"].shape, f["session_start_time"][()].decode()

from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(32) as ex:
    heads = list(ex.map(strobe_header, assets["001274"]))
strobe = pd.DataFrame([dict(subject=a["path"].split("/")[0], path=a["path"], url=a["url"],
                            kind=strobe_kind(a["description"])[0], volts=strobe_kind(a["description"])[1],
                            solution=strobe_kind(a["description"])[2], frames=h[0][0],
                            shape=h[0], start=h[1]) for a, h in zip(assets["001274"], heads)])
print(strobe.groupby(["subject", "kind", "volts", "solution", "frames"]).size().to_string())
print("image sizes:", strobe["shape"].apply(lambda s: s[1:]).value_counts().to_dict())

The strobe files hold the raw image stacks only: no ROIs, no RATIO images, no field per session (the
descriptions give generator volts, not V/cm) and no frame times. The paper's nine cells C1 to C9 each had
a 2-us pulse and 100-Hz modulated sine waves; nine subjects here have a pulse and 100-Hz sessions
(listed below), which is consistent with them being C1 to C9 but does not show it. The cell of Figure 5
(C7) had 12 sessions at each of 100 and 154 Hz; no subject has 12 sessions at 100 Hz.
Many sessions recorded on 2024-08-24 are labelled as made in 1 uM tetrodotoxin, which the paper does not
mention for Figures 5, 6 or S3.

In [ ]:
has = strobe.groupby("subject").kind.agg(lambda k: set(k))
cand = sorted(s for s, k in has.items() if "pulse" in k and "sine 100 Hz" in k)
print("subjects with a pulse and 100-Hz sessions:", len(cand), cand)
c = strobe[(strobe.kind != "pulse")].groupby(["subject", "kind"]).size().unstack(fill_value=0)
print("most 100-Hz sessions in one subject:", c.get("sine 100 Hz", pd.Series([0])).max(),
      "; most 154-Hz sessions:", c.get("sine 154 Hz", pd.Series([0])).max())

### DANDI:001034 is a second deposit of 001032

The two dandisets have the same subjects (`sub-N-1` in 001032, `sub-N-3` in 001034), the same sessions
and the same sizes. The check below compares asset names (some differ in their session part), matches every
asset by its session start time and size, and compares the image arrays of ten pairs.

In [ ]:
def strip(p, k):
    return re.sub(rf"sub-(\d+)-{k}", r"sub-\1", p.split("/")[1])
a32 = {strip(x["path"], 1): x for x in assets["001032"]}
a34 = {strip(x["path"], 3): x for x in assets["001034"]}
print("asset names in common:", len(set(a32) & set(a34)), "of", len(a32), "and", len(a34))
same_meta = sum(a32[k]["session"] == a34[k]["session"] and a32[k]["size"] == a34[k]["size"]
                for k in set(a32) & set(a34))
print("same session name and size:", same_meta)
k32 = {(x["start_date"], x["size"]) for x in assets["001032"]}
k34 = {(x["start_date"], x["size"]) for x in assets["001034"]}
print("assets matched by session start time (archive metadata) and size:", len(k32 & k34), "of", len(k32), "and", len(k34),
      f"({len(k32)} and {len(k34)} distinct pairs)")
def stack_and_start(url):
    with h5py.File(remfile.File(url), "r") as f:
        g = f["acquisition/TimeLapseImaging"]
        return g["data"][:], f["session_start_time"][()].decode()
common = sorted(set(a32) & set(a34))
pairs = common[:: max(1, len(common) // 10)][:10]
identical = 0
for k in pairs:
    x, sx = stack_and_start(a32[k]["url"]); y, sy = stack_and_start(a34[k]["url"])
    identical += int(x.shape == y.shape and np.array_equal(x, y) and sx == sy)
print(f"identical arrays and start times: {identical} of {len(pairs)} pairs compared")

## 2. Load the fast recordings

Every fast recording of 001032, 001033, 001035 (Figures 1 and 2), 001030 (Figure 3C), 001271 (Figure S2)
and 001272 (Figure S1) is streamed and reduced to two dF traces as it loads (`ti_sine.load_fast`). 001034 is
left out as a duplicate. This takes about ten minutes.

In [ ]:
FAST = ["001032", "001033", "001035", "001271", "001272"]
recs = {}
for d in FAST:
    t = time.time()
    recs[d] = T.load_fast(assets[d])
    print(d, len(recs[d]), f"files in {time.time() - t:.0f} s")
recs["001030"] = T.load_fast(assets["001030"], stim_start_s=0.025)   # 001030: stimulus at 25 ms
print("001030", len(recs["001030"]))

## 3. Data check

In [ ]:
fast = pd.concat([T.session_table(recs[d], d) for d in ["001032", "001033", "001035"]], ignore_index=True)
t71 = T.session_table(recs["001271"], "001271")
t72 = T.session_table(recs["001272"], "001272", invert=True, sd_levels=(5,))
allrec = [r for d in recs for r in recs[d]]
byp = {(d, r["path"]): r for d in recs for r in recs[d]}
print("files:", {d: len(v) for d, v in recs.items()})
print("frame shapes and rates:")
print(pd.DataFrame([dict(dandiset=d, shape=str(r["shape"]), rate=r["rate"]) for d in recs for r in recs[d]])
      .groupby(["dandiset", "shape", "rate"]).size().to_string())
print("\nneurons per dandiset:", {d: len({T.neuron_of(d, r['path']) for r in recs[d]}) for d in recs})
print("\nsessions by carrier and modulation (001032, 001033, 001035):")
print(fast.groupby(["dandiset", "carrier", "modulated"]).size().unstack(fill_value=0).to_string())
print("\ngenerator voltage (V peak to peak), range by dandiset:",
      {d: (g.voltage.min(), g.voltage.max()) for d, g in fast.groupby("dandiset")})
ff = np.array([r["first_frame_ratio"] for r in allrec])
print(f"\nfirst frame over the mean of the other pre-stimulus frames: median {np.median(ff):.2f}, "
      f"range {ff.min():.2f} to {ff.max():.2f} (left out of F0)")
print("raw F0 (camera counts), range:", f"{min(r['raw_f0'] for r in allrec):.0f} to {max(r['raw_f0'] for r in allrec):.0f}")
for roi in T.ROIS:
    sd = fast[f"pre_sd/{roi}"]
    print(f"pre-stimulus SD of dF ({roi}): median {sd.median():.2f}%, range {sd.min():.2f} to {sd.max():.2f}")

Checks of the descriptions against the data:

- The paper's fast recordings have 64 x 64 pixels and 1,200 frames per session at 3,130 or 3,086.4 frames/s
  (Methods). The files have 16 x 16 pixels; most have 1,200 frames at 3,086.4 frames/s, some 900, 1,400
  or 1,800 (001032) and one 6,000 (a long sham, 001035). The 2-s recordings of 001030 have 6,000 or 7,000
  frames at 3,134.8 frames/s; at 6,000 frames the recording ends before the 2-s stimulus does.
- The description of 001032 to 001035 gives a 140-um electrode gap; the Methods give 120 um.
- AP size: the Methods give typical APs of 11.3% +- 0.9% dF. The first AP of each session, whole-frame
  trace, is printed below and serves to choose the soma reading.
- Stimulus timing: the files carry no stimulus channel. Two checks support the 50-ms onset of the Methods:
  APs are found during the stimulus and almost never before it, and in sessions with 20-Hz modulation the
  APs lock to the beats counted from that onset, which they do not when the beats are shifted by half a
  beat (25 ms).

In [ ]:
for roi in T.ROIS:
    h = []
    for d in ["001032", "001033", "001035"]:
        for r in recs[d]:
            a = T.aps_in_window(r[roi], r["rate"], r["onset"], reading="a")
            if len(a):
                h.append(r[roi][r["onset"] + int(round(a[0] * r["rate"]))])
    print(f"first-AP peak dF, {roi} trace: median {np.median(h):.1f}%, IQR {np.percentile(h, 25):.1f} to "
          f"{np.percentile(h, 75):.1f}%, n = {len(h)} sessions")
for key in [f"{roi}/{rd}" for roi in T.ROIS for rd in T.AP_READINGS]:
    pre = fast[f"pre_aps/{key}"].sum()
    dur = fast[f"naps/{key}"].sum()
    print(f"{key}: {pre} APs in {len(fast)} pre-stimulus windows (49 ms each; {pre / len(fast) * 250 / 49:.2f} "
          f"per 250 ms) against {dur} during the 250-ms stimuli ({dur / len(fast):.2f} per session)")

### Testing the AP detection

The verifier of the first round showed that reading (a), as first written (a peak above 5% with a prominence
of 2.5%), counts noise on a depolarized plateau as APs. Reading (a) now also asks the peak to rise 5% above
the trace on both sides (prominence) and to be at least 1 ms wide at half its prominence (`ti_sine`, from the
Methods' AP size and duration; an assumption). The cell below runs both versions of reading (a), and reading (b),
on synthetic traces with the data's median pre-stimulus noise (0.83% SD): a plateau without APs, and trains of
APs of the Methods' typical size (11.3%) at 40 to 100 Hz, with and without a plateau under them. It then shows,
for the peaks the first version found in the data, the half-widths and prominences of first APs (clear APs)
and of the other peaks in sessions that end depolarized.

In [ ]:
from scipy.signal import find_peaks, peak_widths
rng = np.random.default_rng(11)
rate = 3086.4; on = int(round(0.05 * rate)); n_st = int(round(0.25 * rate)); tt = np.arange(1200) / rate
def synth_ap(t0, peak=11.3, tau=0.003, rise=0.0007):
    s = np.where(tt >= t0, np.exp(-(tt - t0) / tau) * (1 - np.exp(-(tt - t0) / rise)), 0.0)
    return peak * s / s.max()
def plateau(level):
    return np.r_[np.zeros(on), np.full(n_st, level), np.zeros(1200 - on - n_st)]
rules = {"(a) first round": dict(reading="a", prominence=2.5, min_width_s=0.0),
         "(a) revised": dict(reading="a"), "(b)": dict(reading="b")}
def count(x, kw):
    p = T.detect_aps(x, rate, **kw); p = p[(p >= on) & (p < on + n_st)]
    return len(p), (rate / (p[1] - p[0]) if len(p) > 1 else np.nan)
print("plateau without APs, 200 traces each: APs per 250 ms (mean, max) and median first-two-AP rate")
for lev in (4.0, 6.0):
    noise = [rng.normal(0, 0.83, 1200) for _ in range(200)]
    for name, kw in rules.items():
        c = np.array([count(plateau(lev) + z, kw) for z in noise])
        print(f"  plateau {lev:.0f}%, {name}: {c[:, 0].mean():.1f} (max {c[:, 0].max():.0f}); "
              f"rate {np.nanmedian(c[:, 1]) if np.isfinite(c[:, 1]).any() else np.nan:.0f} Hz")
print("trains of APs (11.3%, decay 2 or 4 ms) from 10 ms after onset: APs found of those given, first-two-AP rate")
for hz in (40, 60, 80, 100):
    for lev in (0.0, 4.0):
        out = []
        for tau in (0.002, 0.004):
            x = plateau(lev) + sum(synth_ap(0.06 + k / hz, tau=tau) for k in range(int(0.2 * hz))) + rng.normal(0, 0.83, 1200)
            for name in ("(a) revised", "(b)"):
                n, r_ = count(x, rules[name])
                out.append(f"{name} {tau * 1000:.0f} ms: {n}/{int(0.2 * hz)}, {r_:.0f} Hz")
        print(f"  {hz} Hz on {lev:.0f}%: " + "; ".join(out))
# the peaks the first-round rule found in the data
pk = []
for _, s in fast.iterrows():
    x = byp[(s.dandiset, s.path)]["whole"]
    p, pr = find_peaks(x, height=5, distance=int(round(T.AP_MERGE_S * s.rate)), prominence=2.5)
    m = (p >= s.onset) & (p < s.onset + int(round(0.255 * s.rate)))
    if not m.any():
        continue
    w = peak_widths(x, p[m], rel_height=0.5)[0] / s.rate * 1000
    for k, (wi, pi) in enumerate(zip(w, pr["prominences"][m])):
        pk.append(dict(first=k == 0, plateau=s["last100/whole"] > 4, width=wi, prom=pi))
pk = pd.DataFrame(pk)
for lab, g in (("first AP of a session", pk[pk.first]), ("other peaks, sessions ending above 4%", pk[~pk.first & pk.plateau]),
               ("other peaks, other sessions", pk[~pk.first & ~pk.plateau])):
    print(f"{lab}: n = {len(g)}, half-width under 1 ms {np.mean(g.width < 1):.2f}, prominence under 5% {np.mean(g.prom < 5):.2f}, "
          f"half-width 1st percentile {g.width.quantile(.01):.2f} ms, prominence 1st percentile {g.prom.quantile(.01):.1f}%")

In [ ]:
def item(s, roi="whole"):
    """A fast session (a row of a session table) as a trace to plot."""
    r = byp[(s.dandiset, s.path)]
    lab = (f"{s.neuron.split('/')[1]} {s.carrier}{' mod' if s.modulated else ''} {T.field(s.voltage):.0f} V/cm")
    return dict(x=r[roi], rate=r["rate"], onset=r["onset"], dur=T.STIM_DUR_S, label=lab)

def show(items, title, ncols=6, xlim=None):
    """Traces with the APs of reading (a) (red, above) and (b) (blue, below), without the first frame;
    prints the y range."""
    nr = int(np.ceil(len(items) / ncols))
    fig, axs = plt.subplots(nr, ncols, figsize=(2.7 * ncols, 1.9 * nr), squeeze=False)
    lo, hi = np.inf, -np.inf
    for ax, it in zip(axs.flat, items):
        x, rate, on = it["x"], it["rate"], it["onset"]
        t = (np.arange(len(x)) - on) / rate * 1000
        x, t = x.copy(), t.copy()
        x[0] = np.nan                     # the first frame is a camera artifact (left out of F0)
        ax.axvspan(0, it["dur"] * 1000, color="C1", alpha=0.12)
        ax.plot(t, x, lw=0.5, color="0.25")
        na = {}
        for rd, mk, col, off in (("a", "v", "C3", 2.5), ("b", "^", "C0", -2.5)):
            a = T.aps_in_window(it["x"], rate, on, it["dur"], rd)
            i = on + np.round(a * rate).astype(int)
            ax.plot(t[i], x[i] + off, mk, color=col, ms=3, ls="none")
            na[rd] = len(a)
        ax.set_title(f"{it['label']}  a{na['a']} b{na['b']}", fontsize=7)
        ax.tick_params(labelsize=6)
        if xlim:
            ax.set_xlim(*xlim)
        lo, hi = min(lo, float(np.nanmin(x))), max(hi, float(np.nanmax(x)))
    for ax in axs.flat[len(items):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
    print(f"{title}: {len(items)} traces, dF from {lo:.1f} to {hi:.1f}% (all inside the axes); x is ms from stimulus onset")

In [ ]:
mod = fast[fast.carrier.isin(["2k", "20k"]) & fast.modulated]
def in_peak_window(times_s, shift_s=0.0):
    ph = ((times_s - shift_s) * 1000) % 50
    return (ph >= 12.5) & (ph <= 37.5)
for key in ["whole/a", "whole/b"]:
    t = np.concatenate(mod[f"aps/{key}"].values)
    print(f"{key}: {len(t)} APs in {len(mod)} modulated sessions; in the middle half of the beat: "
          f"{in_peak_window(t).mean():.2f} with beats from the 50-ms onset, "
          f"{in_peak_window(t, 0.025).mean():.2f} with beats shifted by 25 ms (chance 0.5)")

The whole-frame trace gives APs of the size the Methods report, so it is adopted as the soma reading
(assumption); the bright-pixel trace is shown as a variant throughout. Reading (b) on the bright-pixel
trace finds more than one AP per 250 ms in the pre-stimulus windows, so it is the noisiest of the four.

The field: the descriptions of 001032 to 001035, 001271 and 001272 say "1V translates into 36 V/cm at the
location of neurons" and give peak-to-peak volts; 001030 gives volts of one polarity. The claims file takes
the field from this metadata. The adopted reading is the peak field, 36 V/cm x Vpp / 2 (assumption: it is
how 001030 states its voltages, and Figure 3A's 63 V/cm near threshold matches the 1.75 V of one polarity
there), and 36 V/cm x Vpp is run as the other reading. Section 8 prints this check: the paper's 63 and 81 V/cm
are 3.5 and 4.5 Vpp under the adopted reading, voltages the 2-kHz sessions use, and 1.75 and 2.25 Vpp under the
other, which no session uses. The descriptions also say the generator voltage is
"not the actual amplitude measured on the oscilloscope, which will be somewhat smaller (calibration factors
are frequency-dependent and will be reported separately)"; those factors are not in the dandisets.

## 4. fig1-pure-sine-excites-all

> Unmodulated 250-ms sine waves at 2 kHz and at 20 kHz excited every neuron tested, never as an on-off
> response.

Per neuron, excited when at least one session with an unmodulated 2- or 20-kHz sine wave has an AP during
the stimulus. Reproduced when, under each AP reading, at least 90% of neurons tested with each frequency
are excited. The Methods select neurons whose APs peak above 5% dF; the neurons not excited are checked
for APs in their other sessions, and their traces are shown.

In [ ]:
unmod = fast[fast.carrier.isin(["2k", "20k"]) & ~fast.modulated].copy()
for key in ["whole/a", "whole/b", "bright/a", "bright/b"]:
    e = unmod.groupby(["carrier", "neuron"])[f"naps/{key}"].max() > 0
    res = {c: f"{int(g.sum())}/{len(g)} = {100 * g.mean():.0f}%" for c, g in e.groupby(level=0)}
    print(f"{key}: excited {res}")
e = unmod.groupby(["carrier", "neuron"])["naps/whole/a"].max()
for (c, n) in e[e == 0].index:
    g = unmod[(unmod.neuron == n) & (unmod.carrier == c)]
    print(f"not excited at {c}: {n}, {len(g)} sessions, fields {T.field(g.voltage.min()):.0f} to "
          f"{T.field(g.voltage.max()):.0f} V/cm; APs in its other sessions: {int(fast[fast.neuron == n]['naps/whole/a'].sum())}")
print("neurons by dandiset:", unmod.groupby(["dandiset", "carrier"]).neuron.nunique().to_dict())

In [ ]:
# The two neurons never excited at 20 kHz: their 20-kHz sessions, a session where they fired, and their mean images.
nx = ["001033/sub-7-2", "001035/sub-10-4"]
its = []
for n in nx:
    g = unmod[(unmod.neuron == n)].assign(field=T.field(unmod.voltage))
    its += [item(r) for _, r in g[g.carrier == "20k"].sort_values("field").iterrows()]
    f = fast[(fast.neuron == n) & (fast["naps/whole/a"] > 0)]
    its += [item(r) for _, r in f.sort_values("start").head(1).iterrows()]
show(its, "neurons not excited at 20 kHz (last panel of each neuron: a session with APs)")
fig, axs = plt.subplots(1, 2, figsize=(5, 2.4))
for ax, n in zip(axs, nx):
    s0 = fast[fast.neuron == n].iloc[0]
    im = byp[(s0.dandiset, s0.path)]["mean_image"]
    ax.imshow(im, cmap="gray"); ax.contour(im > np.median(im), [0.5], colors="C1", linewidths=0.8)
    ax.set_title(f"{n} (bright half outlined)", fontsize=7); ax.axis("off")
    print(f"{n}: mean pre-stimulus image {im.min():.0f} to {im.max():.0f} counts; bright half over dim half {im[im > np.median(im)].mean() / im[im <= np.median(im)].mean():.2f}")
plt.tight_layout(); plt.show()
allimg = [r["mean_image"] for d in ["001032", "001033", "001035"] for r in recs[d]]
print(f"mean pre-stimulus image (counts), recordings of all neurons: median {np.median([im.mean() for im in allimg]):.0f}, "
      f"range {min(im.mean() for im in allimg):.0f} to {max(im.mean() for im in allimg):.0f}")
for n in nx:
    ims = [byp[(s.dandiset, s.path)]["mean_image"] for _, s in fast[fast.neuron == n].iterrows()]
    print(f"{n}: mean pre-stimulus image {np.median([im.mean() for im in ims]):.0f} counts (median over its {len(ims)} recordings), "
          f"percentile {stats.percentileofscore([im.mean() for im in allimg], np.median([im.mean() for im in ims])):.0f} of all recordings")
print("bright over dim half (contrast), all neurons: median",
      round(float(np.median([np.mean(r["mean_image"][r["mean_image"] > np.median(r["mean_image"])]) / np.mean(r["mean_image"][r["mean_image"] <= np.median(r["mean_image"])])
                             for d in ["001032", "001033", "001035"] for r in recs[d]])), 2))
# small upticks under the 5% level: peaks of 2.5% prominence and 1 ms half-width, with no level over F0
for n in nx:
    g = unmod[(unmod.neuron == n) & (unmod.carrier == "20k")].assign(field=T.field(unmod.voltage)).sort_values("field")
    cnt = []
    for _, r in g.iterrows():
        x = byp[(r.dandiset, r.path)]["whole"]
        p = find_peaks(x, distance=int(round(T.AP_MERGE_S * r.rate)), prominence=2.5, width=(T.MIN_WIDTH_S * r.rate, None))[0]
        cnt.append(f"{r.field:.0f}: {int(((p >= r.onset) & (p < r.onset + int(round(0.25 * r.rate)))).sum())}")
    pre = [byp[(r.dandiset, r.path)] for _, r in g.iterrows()]
    npre = sum(int(((q >= 1) & (q < r["onset"])).sum()) for r in pre
               for q in [find_peaks(r["whole"], distance=int(round(T.AP_MERGE_S * r["rate"])), prominence=2.5, width=(T.MIN_WIDTH_S * r["rate"], None))[0]])
    print(f"{n} at 20 kHz, upticks of 2.5% per session by field (V/cm): {', '.join(cnt)}; in the pre-stimulus windows of these sessions: {npre}")


**Result: not reproduced.** At 2 kHz, 27 of 28 neurons (96%) fired at least one AP under both AP readings,
and at 20 kHz 15 of 17 (88%), below the 90% the criterion asks of each frequency; the bright-pixel trace gives
the same counts except 28 of 28 at 2 kHz under reading (b). The two neurons never excited at 20 kHz were tested
up to 288 and 324 V/cm and fired in other sessions (26 and 11 APs), so the Methods' selection of neurons with
APs does not remove them. Their 20-kHz traces show no AP of 5%. 001035/sub-10-4 has 8 upticks of 2.5% at
324 V/cm and 6 at 216 V/cm, on a falling trace, against 4 in all the pre-stimulus windows of its 20-kHz sessions;
these may be small APs, but no reading of the claims file counts them, and with them the 20-kHz fraction would
be 16 of 17 (94%). 001033/sub-7-2 has 3 to 7 such upticks per session at every field, at its pre-stimulus rate
(8). Neither neuron is dimmer than the others: their mean pre-stimulus images (3110 and 3012 counts) are at the
78th and 74th percentiles of all recordings, and their contrast (bright half over dim half 1.68 and 1.52) is at
or above the median of 1.47. Whether
firing was ever an on-off response is not tested by the criterion.

## 5. fig1-onset-delay

> Near threshold, isolated APs appear 100 to 200 ms after the sine wave onset, and stronger fields shorten
> this delay and evoke sustained firing.

Per neuron and frequency, the near-threshold session is the lowest field with at least one AP. Reproduced
when the median first-AP latency of those sessions lies within 90 to 220 ms and the Spearman correlation of
first-AP latency with field is below zero in more than half of the neuron-frequency pairs with at least
three fields with APs. The latency does not depend on the field reading.

In [ ]:
onset = {}
for key in ["whole/a", "whole/b", "bright/a", "bright/b"]:
    pf = T.per_field(unmod, key)
    nt = pf[pf.n > 0].sort_values("field").groupby(["neuron", "carrier"]).first()
    lat = nt.lat * 1000
    rho = np.array([stats.spearmanr(g.field, g.lat).statistic
                    for _, g in pf[pf.n > 0].groupby(["neuron", "carrier"]) if len(g) >= 3])
    onset[key] = (lat, rho)
    by_c = {c: f"{g.median():.0f} ms (n = {len(g)})" for c, g in lat.groupby(level=1)}
    print(f"{key}: near-threshold first-AP latency median {lat.median():.0f} ms, IQR {lat.quantile(.25):.0f} "
          f"to {lat.quantile(.75):.0f}, n = {len(lat)}; by carrier {by_c}; latency falls with field "
          f"(rho < 0) in {np.sum(rho < 0)} of {len(rho)} (median rho {np.median(rho):.2f})")
# variant: thresholds bracketed by a lower-field session without APs
pf = T.per_field(unmod, "whole/a")
br = []
for (n, c), g in pf.groupby(["neuron", "carrier"]):
    g = g.sort_values("field"); fired = g[g.n > 0]
    if len(fired) and (g.field < fired.field.iloc[0]).any():
        br.append(dict(neuron=n, carrier=c, lat=fired.lat.iloc[0] * 1000))
br = pd.DataFrame(br)
print(f"variant, whole/a, near-threshold sessions with a lower-field session without APs: median "
      f"{br.lat.median():.0f} ms, n = {len(br)}; by carrier {br.groupby('carrier').lat.median().round(0).to_dict()}")
print("by dandiset (whole/a):", onset["whole/a"][0].groupby(lambda i: i[0].split('/')[0]).median().round(0).to_dict())

In [ ]:
lat, rho = onset["whole/a"]
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(lat, bins=np.arange(0, 260, 20), color="0.4")
ax[0].axvspan(100, 200, color="C1", alpha=0.2, label="paper, 100 to 200 ms")
ax[0].set_xlabel("near-threshold first-AP latency (ms)"); ax[0].set_ylabel("neuron-carrier pairs"); ax[0].legend()
ax[1].hist(rho, bins=np.linspace(-1, 1, 11), color="0.4"); ax[1].set_xlabel("Spearman rho, latency against field")
plt.tight_layout(); plt.show()
print(f"latency range {lat.min():.0f} to {lat.max():.0f} ms (all inside the axis); rho range {rho.min():.2f} to {rho.max():.2f}")

In [ ]:
# Near-threshold sessions with a first AP within 20 ms of the stimulus onset (reading a): are they stimulus artifacts?
nt = T.per_field(unmod, "whole/a")
nt = nt[nt.n > 0].sort_values("field").groupby(["neuron", "carrier"]).first().reset_index()
ss = unmod.assign(field=T.field(unmod.voltage)).merge(nt[["neuron", "carrier", "field"]], on=["neuron", "carrier", "field"])
ss = ss[ss["naps/whole/a"] > 0]
ss["first a (ms)"] = ss["aps/whole/a"].apply(lambda a: 1000 * a[0])
ss["first b (ms)"] = ss["aps/whole/b"].apply(lambda a: 1000 * a[0] if len(a) else np.nan)
short = ss[ss["first a (ms)"] < 20].sort_values("first a (ms)")
print(f"near-threshold sessions: {len(ss)}; with a first AP before 20 ms: {len(short)}")
print(np.histogram(short["first a (ms)"], bins=np.arange(0, 21, 2))[0].tolist(), "(first-AP latencies, 2-ms bins from 0 to 20 ms)")
print(short[["neuron", "carrier", "field", "naps/whole/a", "first a (ms)", "first b (ms)"]].round(1).to_string(index=False))
show([item(r) for _, r in short.iterrows()], "near-threshold sessions with a first AP before 20 ms", xlim=(-30, 260))

**Result: not reproduced.** The median first-AP latency of the near-threshold sessions is 64 ms (reading a)
and 67 ms (reading b), below the 90 to 220 ms the criterion asks; it is 49 ms at 2 kHz and 124 ms (a) or 101 ms
(b) at 20 kHz. Restricting to thresholds with a lower-field session without APs gives 67 ms. The second part
holds: the latency falls with field in 38 of 38 (a) and 39 of 39 (b) neuron-carrier pairs. The criterion needs
both parts. Short latencies are not stimulus artifacts: 7 of 50 near-threshold sessions have a first AP within
20 ms; the two earliest (2.6 and 2.9 ms) are at 180 and 153 V/cm, the lowest fields with an AP in those
neurons, and in the traces they are AP-shaped upticks at the start of a depolarization. Of the other five, at 10
to 16 ms, three (001032/sub-11-1, sub-14-1 and sub-10-1) are the first of 4 regular APs, so their lowest tested
field was already above threshold, and two are single APs. The outcome depends on pooling the dandisets, as the
claims file's "where" does: the 001035 neurons alone give a median of 101 ms, inside 90 to 220 ms, and 001032
and 001033 give 50 and 58 ms.

## 6. fig1-max-rate-ceiling

> The maximum firing rate increases linearly with field strength until it reaches a ceiling of 60 to 90 Hz.

The maximum firing rate of a session is the inverse of the interval between its first two APs (Methods);
the ceiling of a neuron is its highest over unmodulated sessions. Reproduced when the median ceiling over
neurons lies within 54 to 99 Hz. The AP detection cannot see rates above 125 Hz (8-ms merge, `ti_sine`).

In [ ]:
for key in ["whole/a", "whole/b", "bright/a", "bright/b"]:
    pf = T.per_field(unmod, key)
    ceil = pf.groupby("neuron").rate_hz.max().dropna()
    slopes = []
    for _, g in pf.dropna(subset=["rate_hz"]).groupby(["neuron", "carrier"]):
        g = g.sort_values("field")
        upto = g.iloc[: int(np.argmax(g.rate_hz.values)) + 1]
        if len(upto) >= 3:
            slopes.append(stats.linregress(upto.field, upto.rate_hz).slope)
    print(f"{key}: ceiling median {ceil.median():.1f} Hz (IQR {ceil.quantile(.25):.1f} to {ceil.quantile(.75):.1f}, "
          f"range {ceil.min():.1f} to {ceil.max():.1f}, n = {len(ceil)} neurons); slope of rate on field before the "
          f"ceiling: median {np.median(slopes):.2f} Hz per V/cm, positive in {np.sum(np.array(slopes) > 0)} of {len(slopes)}")
print("neurons with fewer than two APs in every unmodulated session have no rate:",
      unmod.neuron.nunique() - T.per_field(unmod, 'whole/a').groupby('neuron').rate_hz.max().notna().sum())

In [ ]:
# The sessions that set each neuron's ceiling, under reading (a) or (b), with the APs of both readings marked.
s = unmod.assign(field=T.field(unmod.voltage))
for rd in "ab":
    s[f"rate {rd}"] = s[f"aps/whole/{rd}"].apply(lambda a: 1 / (a[1] - a[0]) if len(a) > 1 else np.nan)
s["stim dF"] = [T.window_mean(byp[(d, p)]["whole"], r, o, 0, 0.25) for d, p, r, o in zip(s.dandiset, s.path, s.rate, s.onset)]
pick = sorted(set(s.dropna(subset=["rate a"]).groupby("neuron")["rate a"].idxmax())
              | set(s.dropna(subset=["rate b"]).groupby("neuron")["rate b"].idxmax()))
ce = s.loc[pick].sort_values(["neuron", "field"])
print(ce[["neuron", "carrier", "field", "naps/whole/a", "naps/whole/b", "rate a", "rate b", "stim dF", "last100/whole"]]
      .round(1).to_string(index=False))
show([item(r) for _, r in ce.iterrows()], "sessions setting a neuron's rate ceiling")

**Result: not reproduced, resting on an assumption.** The median ceiling is 39.1 Hz under reading (a), as
revised, and 42.9 Hz under reading (b), both below 54 to 99 Hz; the bright-pixel trace gives 41.7 (a) and
61.7 Hz (b), the last being the noisiest reading (1.18 false APs per 250 ms before stimuli, section 3). The
traces of the sessions that set each ceiling show the APs found. Half the neurons have a ceiling between 28.8
and 60.5 Hz (IQR, reading a), and 5 of the 35 sessions shown reach 79 to 94 Hz under both readings.
The rate rises with field before the ceiling in 32 of 32 neuron-carrier pairs (median slope 0.66 Hz per V/cm).
The first version of reading (a), without the width and with a 2.5% prominence, gave 59.9 Hz, inside the
range, by counting noise on depolarized plateaus (section 3: about 18 false APs per 250 ms on a 6% plateau,
at a rate near 80 Hz), so the outcome rests on the revised reading (a); reading (b), unchanged, gives the
same outcome.

## 7. fig1-excitation-block

> At the highest field strengths, APs coalesce into low-amplitude oscillations on a continuous
> depolarization, which can flatten and leave the membrane depolarized for the rest of the sine wave (an
> excitation block).

Per neuron and frequency, the highest-field unmodulated session against the near-threshold session.
Reproduced when, in more than half, the mean dF over the last 100 ms of the stimulus at the highest field
is above 5% and above that of the near-threshold session. The mean over the second half of the stimulus is a
variant.

In [ ]:
for key in ["whole/a", "whole/b", "bright/a", "bright/b"]:
    pf = T.per_field(unmod, key)
    nt = pf[pf.n > 0].sort_values("field").groupby(["neuron", "carrier"]).first()
    hi = pf.sort_values("field").groupby(["neuron", "carrier"]).last()
    j = hi.join(nt[["last", "field"]].rename(columns={"last": "last_nt", "field": "field_nt"}), how="inner")
    blk = (j["last"] > 5) & (j["last"] > j["last_nt"])
    blk2 = j["half2"] > 5
    print(f"{key}: block in {blk.sum()} of {len(j)} neuron-carrier pairs; highest-field last-100-ms dF median "
          f"{j['last'].median():.1f}% (range {j['last'].min():.1f} to {j['last'].max():.1f}); near-threshold median "
          f"{j['last_nt'].median():.1f}%; highest over near-threshold field median {np.median(j.index.map(lambda i: hi.loc[i, 'field']) / j.field_nt):.2f}; "
          f"variant second half > 5%: {blk2.sum()}")
d = unmod.assign(field=T.field(unmod.voltage))
print("sessions with last-100-ms dF above 5% (whole):", int((d['last100/whole'] > 5).sum()), "of", len(d),
      "; highest field per carrier:", d.groupby("carrier").field.max().to_dict())

In [ ]:
# The highest-field unmodulated session of each neuron and carrier (the last one recorded at that field).
hs = unmod.assign(field=T.field(unmod.voltage)).sort_values(["field", "start"]).groupby(["neuron", "carrier"]).tail(1)
show([item(r) for _, r in hs.sort_values(["neuron", "carrier"]).iterrows()], "highest-field session of each neuron and carrier")

**Result: not reproduced.** The highest-field session has a mean last-100-ms dF above 5% and above the
near-threshold session in 3 of 42 neuron-carrier pairs under both readings (3 of 42 and 3 of 43 with the
bright trace), far from more than half; the second-half variant also gives 3. The highest field tested was a
median 1.36 times the near-threshold field, and 11 of 370 unmodulated sessions have a last-100-ms dF above 5%.
The traces of the highest-field sessions show sustained firing in most neurons and a flat depolarized plateau
in few. The traces are not corrected for bleaching; the near-threshold sessions end at a median last-100-ms dF
of -0.2%, so the drift is small against the 5% level.

## 8. fig2d-20khz-twofold-threshold, fig2d-modulated-thresholds-higher, threshold-2khz-unmodulated

Thresholds per neuron and stimulus are the lowest field with at least 1 (5) APs (Figure 2D legend).

> All excitation thresholds at 20 kHz were about twice those at 2 kHz (p < 0.0001, two-tailed unpaired t test).

Reproduced when each of the four 20 to 2 kHz ratios of mean thresholds is within 1.8 to 2.2 and the t test
gives p < 0.0001 per group and pooled.

In [ ]:
stim = fast[fast.carrier.isin(["2k", "20k"])]
thr = {(key, fr): T.thresholds(stim, key, fr) for key in ["whole/a", "whole/b", "bright/a", "bright/b"] for fr in ("half", "full")}
ratios = {}
th = thr[("whole/a", "half")]
print("whole/a, field = 36 V/cm x Vpp / 2: mean, SEM and n of thresholds (V/cm)")
print(th.groupby(["aps", "stim"]).thr.agg(["mean", "sem", "count"]).round(1).unstack(0).to_string())
for key in ["whole/a", "whole/b", "bright/a", "bright/b"]:
    th = thr[(key, "half")]
    out = []
    for k in (1, 5):
        for m in ("unmod", "mod"):
            a = th[(th.aps == k) & (th.stim == f"2k/{m}")].thr.dropna()
            b = th[(th.aps == k) & (th.stim == f"20k/{m}")].thr.dropna()
            ratios.setdefault(key, []).append(b.mean() / a.mean())
            out.append(f"{k} AP {m} {b.mean() / a.mean():.2f} (p = {stats.ttest_ind(b, a).pvalue:.1g}, n {len(a)}/{len(b)})")
    a = th[th.stim.str.startswith("2k/")].thr.dropna(); b = th[th.stim.str.startswith("20k/")].thr.dropna()
    out.append(f"pooled {b.mean() / a.mean():.2f} (p = {stats.ttest_ind(b, a).pvalue:.1g})")
    print(f"{key}: 20/2 kHz ratio " + "; ".join(out))

**Result: not reproduced.** Every 20-kHz threshold group is higher than the 2-kHz one with p below 0.0001 (per
group at most 2e-05, pooled 3e-24), but the ratios are 2.27, 2.62, 2.37 and 2.82 (reading a) and 2.29, 2.63,
2.28 and 2.82 (reading b), all above the 1.8 to 2.2 of the criterion, which asks every ratio to be within it.
The ratio does not depend on the field reading. The modulated 20-kHz groups have 2 or 3 neurons. The voltages
are the generator's, uncalibrated for frequency, and the factors are not deposited. A 20-kHz factor relative
to the 2-kHz one scales all four ratios alike, and the ratios differ among themselves by a factor of 1.24, more
than the 1.22 the band 1.8 to 2.2 allows: the factor would have to be at least 0.794 and at most 0.779 (reading
a; 0.791 and 0.779 under b), as the calibration check in the next section prints. No calibration of the
generator voltages can bring all four ratios into the band.

> Thresholds for sine waves modulated at 20 Hz were significantly higher than for pure sine waves of the
> same carrier, for 5 APs at 2 kHz and for 1 and 5 APs at 20 kHz.

Two-tailed unpaired t test per pair; reproduced when all three have the modulated mean higher with p < 0.05,
partially when one or two do. The paired test on neurons with both is a variant; the 2-kHz 1-AP pair is
reported.

In [ ]:
for key in ["whole/a", "whole/b", "bright/a", "bright/b"]:
    th = thr[(key, "half")]
    out = []
    for c, k in [("2k", 5), ("20k", 1), ("20k", 5), ("2k", 1)]:
        u = th[(th.aps == k) & (th.stim == f"{c}/unmod")].set_index("neuron").thr
        m = th[(th.aps == k) & (th.stim == f"{c}/mod")].set_index("neuron").thr
        both = pd.concat([u, m], axis=1, keys=["u", "m"]).dropna()
        pp = stats.ttest_rel(both.m, both.u).pvalue if len(both) > 1 else np.nan
        out.append(f"{c} {k} AP: mod {m.mean():.0f} vs unmod {u.mean():.0f} V/cm, unpaired p = "
                   f"{stats.ttest_ind(m.dropna(), u.dropna()).pvalue:.3f} (n {m.count()}/{u.count()}); paired p = {pp:.3f} (n {len(both)})")
    print(key + ":\n  " + "\n  ".join(out))

**Result: partially reproduced.** The modulated mean is higher in all three pairs under both readings. Under
reading (a) two pairs reach p < 0.05 (20 kHz 1 AP p = 0.021, 20 kHz 5 APs p = 0.017; 2 kHz 5 APs p = 0.069),
and under reading (b) the same two (p = 0.021 and 0.012; 2 kHz 5 APs p = 0.155). The 20-kHz modulated groups
have 2 or 3 neurons. The 2-kHz 1-AP pair, for which the paper reports no difference, gives p = 0.059 (a) and
0.045 (b) unpaired, and p below 0.001 paired on 12 neurons.

> The AP threshold for unmodulated 2-kHz sine waves was 57.7 plus or minus 3.6 V/cm.

Mean 1-AP threshold over neurons, under both AP readings and both field readings.

In [ ]:
for key in ["whole/a", "whole/b", "bright/a", "bright/b"]:
    for fr in ("half", "full"):
        th = thr[(key, fr)]
        u = th[(th.aps == 1) & (th.stim == "2k/unmod")]
        ub = u[u.bracketed]
        print(f"{key}, field {fr}: {u.thr.mean():.1f} +- {u.thr.sem():.1f} V/cm (n = {u.thr.count()}); "
              f"only thresholds with a lower-field session without APs: {ub.thr.mean():.1f} (n = {len(ub)})")
u = thr[("whole/a", "half")]; u = u[(u.aps == 1) & (u.stim == "2k/unmod")]
print("by dandiset (whole/a, half):", u.groupby(u.neuron.str[:6]).thr.agg(["mean", "count"]).round(1).to_dict("index"))

In [ ]:
# The field reading against the paper's numbers: 63 V/cm near threshold at 2 kHz (Figure 3A) and 81 V/cm.
u2 = unmod[unmod.carrier == "2k"]
print("2-kHz unmodulated sessions per generator voltage (V peak to peak):", u2.voltage.value_counts().sort_index().to_dict())
t1 = thr[("whole/a", "half")]; t1 = t1[(t1.aps == 1) & (t1.stim == "2k/unmod")]
print("1-AP thresholds, 2 kHz unmodulated, in V peak to peak:", (t1.thr / 18).round(2).value_counts().sort_index().to_dict())
for vcm in (57.7, 63.0, 81.0):
    vh, vf = vcm / 18, vcm / 36
    print(f"{vcm} V/cm is {vh:.2f} Vpp under the half reading and {vf:.2f} Vpp under the full one; 2-kHz sessions at "
          f"these voltages: {int((u2.voltage.round(2) == round(vh, 2)).sum())} and {int((u2.voltage.round(2) == round(vf, 2)).sum())}; "
          f"lowest voltage recorded {u2.voltage.min()} Vpp")
# calibration: the descriptions say the generator voltage overstates the amplitude; the factors are not deposited
print(re.search(r"Note: this is not[^)]*\)", assets["001035"][0]["description"]).group(0))
note = re.search(r"Note: For 40V.*?measured was \d+V\.", assets["001273"][0]["description"]).group(0)
cal = [(float(a), float(b)) for a, b in re.findall(r"(\d+)V(?: pulse amplitude)?, the actual measured (?:amplitude )?was (\d+)V", note)]
print("DANDI:001273 (2-us pulses, another electrode array):", note)
print("measured over generator amplitude:", [round(b / a, 2) for a, b in cal])
m2 = t1.thr.mean()
print(f"a 2-kHz factor of {57.7 / m2:.2f} would make the mean 1-AP threshold 57.7 V/cm (it is {m2:.1f})")
# one 20-kHz factor f (relative to the 2-kHz one) scales the four 20/2-kHz ratios alike: all are in 1.8 to 2.2
# only if f >= 1.8 / smallest ratio and f <= 2.2 / largest ratio
for key in ("whole/a", "whole/b"):
    r = np.array(ratios[key])
    lo, hi = 1.8 / r.min(), 2.2 / r.max()
    print(f"{key}: ratios {r.min():.2f} to {r.max():.2f}, spread {r.max() / r.min():.2f} (the band 1.8 to 2.2 allows {2.2 / 1.8:.2f}); "
          f"the 20-kHz factor would have to be at least {lo:.3f} and at most {hi:.3f}: "
          f"{'possible' if lo <= hi else 'no single factor brings all four into the band'}")

**Result: not reproduced.** The mean 1-AP threshold for unmodulated 2-kHz sine waves is 75.6 +- 6.2 V/cm
(n = 27, reading a) and 74.8 +- 5.9 (reading b) with the field as 36 V/cm times half the peak-to-peak voltage,
and 151.2 and 149.6 V/cm with the whole of it; none is within 57.7 +- 3.6. By dandiset it is 92.2 (001032),
76.6 (001033) and 58.3 V/cm (001035). The half reading is supported by the voltages used: the paper's 63 and
81 V/cm are 3.5 and 4.5 Vpp under it (2 and 7 sessions) and 1.75 and 2.25 Vpp under the full reading (no
session). The generator voltage overstates the amplitude by a frequency-dependent factor that is not
deposited. A 2-kHz factor of 0.76 would bring the mean to 57.7 V/cm; the only measured factors in the paper's
dandisets, for 2-us pulses with another electrode array (001273), are 0.62 to 0.79. The miss may therefore
come from the calibration, which cannot be applied here. The outcome also depends on pooling the dandisets, as
the claims file's "where" does: the 001035 neurons alone give 58.3 V/cm, inside 57.7 +- 3.6, and as 001035 has
no modulated sessions, Figure 2D cannot come from it alone, so which neurons the paper used cannot be settled.

## 9. fig2-modulated-locks-to-beats

> With 20-Hz modulation, firing above threshold is locked to the beats, one AP near the peak of each of the
> five beats and at most two per beat (ten in all), and modulated sine waves trigger no sustained firing or
> sustained depolarization.

Reproduced when (i) no modulated session has more than 10 APs, (ii) in sessions with 5 APs or more at least
80% of APs fall 12.5 to 37.5 ms into a 50-ms beat counted from the onset, and (iii) no modulated session has
a last-100-ms mean dF above 5% and above that of the neuron's near-threshold modulated session (the block
test of fig1-excitation-block). (ii) is read on the APs pooled over those sessions; the fraction of sessions
that pass on their own is also shown.

In [ ]:
for key in ["whole/a", "whole/b", "bright/a", "bright/b"]:
    roi = key.split("/")[0]
    n = mod[f"naps/{key}"]
    sel = mod[n >= 5]
    pooled = np.concatenate(sel[f"aps/{key}"].values)
    per = np.array([in_peak_window(a).mean() for a in sel[f"aps/{key}"]])
    hist = np.histogram((np.concatenate(mod[f"aps/{key}"].values) * 1000) % 50, bins=10, range=(0, 50))[0]
    print(f"{key}: {len(mod)} sessions; most APs {n.max()}, sessions above 10: {(n > 10).sum()}; "
          f"{len(sel)} sessions with >= 5 APs, pooled fraction in window {in_peak_window(pooled).mean():.2f}, "
          f"sessions with >= 80%: {np.sum(per >= 0.8)} of {len(sel)}; sessions with last-100-ms dF > 5%: "
          f"{(mod[f'last100/{roi}'] > 5).sum()}; APs per 5 ms of the beat {hist.tolist()}")
print(mod.assign(field=T.field(mod.voltage)).loc[(mod["naps/whole/a"] > 10) | (mod["last100/whole"] > 5),
      ["neuron", "carrier", "field", "naps/whole/a", "naps/whole/b", "last100/whole"]].to_string())

In [ ]:
# Part (iii) with the whole block test of fig1-excitation-block: last-100-ms dF above 5% and above that of the
# near-threshold modulated session of the same neuron and carrier.
pm = T.per_field(mod, "whole/a")
ntm = pm[pm.n > 0].sort_values("field").groupby(["neuron", "carrier"]).first()
hot = mod.assign(field=T.field(mod.voltage))
hot = hot[hot["last100/whole"] > 5]
for _, r in hot.iterrows():
    ref = ntm.loc[(r.neuron, r.carrier)]
    print(f"{r.neuron} {r.carrier} at {r.field:.0f} V/cm: last-100-ms dF {r['last100/whole']:.1f}%; near-threshold modulated "
          f"session {ref.field:.0f} V/cm, {ref['last']:.1f}%; meets the block test: {bool(r['last100/whole'] > 5 and r['last100/whole'] > ref['last'])}")
top = mod.assign(nmax=mod[["naps/whole/a", "naps/whole/b"]].max(axis=1)).sort_values("nmax").tail(3)
show([item(r) for _, r in pd.concat([top, mod.loc[hot.index]]).iterrows()], "modulated sessions with the most APs, and the one ending above 5%")

**Result: not reproduced.** Firing locks to the beats: of the APs in modulated sessions with 5 APs or more,
0.96 (reading a) and 0.98 (reading b) fall in the middle half of a beat, against 0.04 of the APs pooled over
all modulated sessions when the beats are shifted by 25 ms, so (ii) holds. (i) holds under both readings (at
most 8 and 6 APs). (iii) fails under both: one session (001032/sub-2-1, 198 V/cm) ends with a
last-100-ms dF of 5.9%, above its near-threshold modulated session (-0.2% at 144 V/cm), and its trace shows
the plateau after the third beat. The criterion asks all three parts.

## 10. fig2c-pure-more-aps

> In the same neuron at the same field strength, pure sine waves trigger more APs than modulated ones.

Pairs are a neuron's modulated and unmodulated sessions at one carrier and one field (counts averaged over
repeats). Reproduced when the unmodulated count is higher in more pairs than the reverse, with a two-sided
paired t test at p < 0.05.

In [ ]:
for key in ["whole/a", "whole/b", "bright/a", "bright/b"]:
    g = stim.groupby(["neuron", "carrier", "voltage", "modulated"])[f"naps/{key}"].mean().unstack("modulated").dropna()
    d = g[False] - g[True]
    print(f"{key}: {len(g)} pairs in {g.reset_index().neuron.nunique()} neurons; unmodulated more in {(d > 0).sum()}, "
          f"modulated more in {(d < 0).sum()}, equal in {(d == 0).sum()}; mean difference {d.mean():.2f} APs; "
          f"paired t p = {stats.ttest_rel(g[False], g[True]).pvalue:.2g}; by carrier "
          f"{d.groupby(level=1).agg(lambda x: f'{(x > 0).sum()}/{len(x)}').to_dict()}")

**Result: reproduced.** In 52 pairs of sessions from 12 neurons at the same carrier and field, the
unmodulated sine wave gave more APs in 47 and fewer in none (5 equal), a mean of 3.70 more APs (reading a,
paired t p = 7.3e-10; reading b 46 and 1, p = 2.5e-09).

## 11. fig3c-sustained-firing-2s (DANDI:001030)

> In the low-conductivity solution, a 2-s unmodulated 2-kHz sine wave ramped up over 250 ms evoked sustained
> firing for the whole stimulus; firing rate rose with field strength until replaced by a firing block with
> sustained depolarization, which developed before the end of the ramp.

Traces are corrected for bleaching by subtracting a trace without APs (Methods). The only trace without APs
in 001030 is neuron c7 at 1.0 V (one polarity, 36 V/cm), so the test is made on c7, the neuron recorded at the
most fields, and the other neurons are shown with the same correction. Reproduced when (i) at an intermediate
field APs occur in the last 250 ms, (ii) the AP count rises with field up to some field, and (iii) at the
highest field there are no APs after 250 ms and the mean dF over the rest of the stimulus is above 5%.

In [ ]:
r30 = {r["path"]: r for r in recs["001030"]}
t30_raw = T.session_table(recs["001030"], "001030", dur_s=2.0)
noap = t30_raw[(t30_raw["naps/whole/a"] == 0) & (t30_raw["naps/whole/b"] == 0)]
print("sessions without APs under both readings (uncorrected):", noap[["neuron", "voltage"]].values.tolist())
ref = r30[noap.path.iloc[0]]
corr = {}
for p, r in r30.items():
    n = min(len(r["whole"]), len(ref["whole"]))
    corr[p] = {roi: r[roi][:n] - ref[roi][:n] for roi in T.ROIS}
t30 = T.session_table(recs["001030"], "001030", dur_s=2.0, traces=corr)
t30["field"] = T.field(t30.voltage)          # voltage holds 2 x the one-polarity volts
rows = []
for _, s in t30.iterrows():
    rate, on = s.rate, s.onset
    n = len(corr[s.path]["whole"])
    row = dict(neuron=s.neuron, field=s.field, frames=s.n_frames, rest_dF=round(s["half2/whole"], 1))
    end = min(2.0, (n - on) / rate)
    row["rest_dF"] = round(T.window_mean(corr[s.path]["whole"], rate, on, 0.25, end), 1)
    for rd in "ab":
        a = s[f"aps/whole/{rd}"]
        row[f"{rd}: all"] = len(a); row[f"{rd}: after 250 ms"] = int((a > 0.25).sum()); row[f"{rd}: last 250 ms"] = int((a > 1.75).sum())
    rows.append(row)
f3c = pd.DataFrame(rows).sort_values(["neuron", "field"])
print(f3c.to_string(index=False))
c7 = f3c[f3c.neuron.str.endswith("c7")]
top = c7.iloc[-1]
a_top = t30[(t30.neuron == top.neuron) & (t30.field == top.field)]["aps/whole/b"].iloc[0]
print(f"c7 at the highest field ({top.field:.0f} V/cm), reading b AP times (s): {np.round(a_top, 3).tolist()}")

In [ ]:
ptop = t30[(t30.neuron == top.neuron) & (t30.field == top.field)].iloc[0]
print("c7 at the highest field, reading (a) AP times after 250 ms (s):",
      np.round(ptop["aps/whole/a"][ptop["aps/whole/a"] > 0.25], 3).tolist())
c7s = t30[t30.neuron.str.endswith("c7")].sort_values("field")
show([dict(x=corr[p]["whole"], rate=r30[p]["rate"], onset=r30[p]["onset"], dur=2.0, label=f"c7 {f:.0f} V/cm")
      for p, f in zip(c7s.path, c7s.field)], "neuron c7 of 001030, bleaching-corrected", ncols=3)

In [ ]:
# Part (iii): are the peaks after 250 ms at the highest field more than the noise of the corrected trace gives?
# The trace's slow course is a 31-ms running median; its residual is the noise. Three estimates of the number
# of false APs over 0.25 to 2 s: the same trace mirrored about its running median (upward noise events become
# downward ones and the reverse, so detections there come from noise of this trace), and the running median
# plus Gaussian noise of the residual's SD, or plus the residual shuffled in time (300 draws each).
from scipy.ndimage import median_filter
x = corr[ptop.path]["whole"]; rate_, on_ = r30[ptop.path]["rate"], r30[ptop.path]["onset"]
i0, i1 = on_ + int(round(0.25 * rate_)), on_ + int(round(2.0 * rate_))
base = median_filter(x, size=int(round(0.03 * rate_)) | 1, mode="nearest")
res = x[i0:i1] - base[i0:i1]
print(f"c7 {ptop.field:.0f} V/cm: residual SD on the plateau {res.std():.2f}% (pre-stimulus SD of the corrected trace "
      f"{x[1:on_].std():.2f}%, of the fast recordings 0.83%)")
def n_after(y, rd):
    p = T.detect_aps(y, rate_, rd)
    return int(((p >= i0) & (p < i1)).sum())
rng = np.random.default_rng(5)
noise_rate = {}
for rd in "ab":
    obs = n_after(x, rd)
    mir = n_after(2 * base - x, rd)
    g = np.array([n_after(base + rng.normal(0, res.std(), len(x)), rd) for _ in range(300)])
    sh = []
    for _ in range(300):
        y = base.copy(); y[i0:i1] += rng.permutation(res); sh.append(n_after(y, rd))
    sh = np.array(sh)
    noise_rate[rd] = g.mean() / 7
    print(f"reading ({rd}): {obs} peaks after 250 ms; mirrored trace {mir}; Gaussian noise mean {g.mean():.1f} "
          f"(5th to 95th percentile {np.percentile(g, 5):.0f} to {np.percentile(g, 95):.0f}, draws with {obs} or more {np.mean(g >= obs):.2f}); "
          f"shuffled residual mean {sh.mean():.1f} (draws with {obs} or more {np.mean(sh >= obs):.2f})")
print("false APs per 250 ms on this plateau (Gaussian):", {k: round(v, 2) for k, v in noise_rate.items()},
      "; APs in the last 250 ms at 52, 61 and 72 V/cm (part i):",
      c7[["field", "a: last 250 ms", "b: last 250 ms"]].iloc[2:5].values.tolist())


**Result: not reproduced (change to reproduced proposed).** In neuron c7, (i) APs occur in the last 250 ms at
52, 61 and 72 V/cm (8 to 10 under both readings), and (ii) the AP count rises with field, from 0 at 36 V/cm to
89 (a) or 90 (b) at 61 V/cm, then falls to 88 and 13 (a) or 80 and 8 (b). At the highest field, 83 V/cm, the
mean dF over 0.25 to 2 s is 6.7%, above 5%, and the APs stop within the ramp, but reading (a) finds 5 peaks after
250 ms (at 0.34, 0.70, 1.31, 1.62 and 1.74 s) and reading (b) one (1.68 s). Taken literally, (iii) asks no APs
after 250 ms, and it fails under both readings; this is the outcome the second verification round reviewed.

The verifier of the second round argued that these peaks are at the detectors' false-positive rate. The cell
above measures that rate on this trace. Its noise on the plateau has an SD of 0.94%; the same trace mirrored
about its running median gives 6 (a) and 7 (b) detections over 0.25 to 2 s, and its running median plus
Gaussian noise of that SD gives a mean of 7.8 (a) and 5.1 (b), with 5 or more in 91% of draws under (a) and 1
or more in 99% under (b); the shuffled residual gives 8.4 and 4.6. The 5 and 1 peaks found are therefore no
more than the noise of the corrected trace produces, and are not evidence of APs. Under the same noise, about
1.1 (a) and 0.7 (b) false APs fall in a 250-ms window, against the 8 to 10 APs of part (i), which stand. Read as
"no more detections after 250 ms than the trace's noise gives" (an assumption), (iii) holds, and so do (i) and
(ii): the change of outcome to reproduced, resting on that assumption, is proposed for a third verification
round. The other three neurons have no AP-free trace of their own and none reaches a block.


## 12. figs1-di8-anepps-control (DANDI:001272)

> With Di-8-ANEPPS instead of FluoVolt, in eight neurons, stimulation near threshold elicited one or a few
> delayed APs and stronger fields induced sustained firing ending in sustained depolarization, as with
> FluoVolt.

Traces inverted (Di-8-ANEPPS emission falls with depolarization, Methods). AP readings: (a) and (b) at 5% dF,
and both at 5 pre-stimulus SDs. Reproduced when, under every reading, every neuron is excited, latency falls
with field in more than half, and the highest field meets the block test in more than half; blocked as
methods_underspecified when the readings disagree.

In [ ]:
print("neurons:", t72.neuron.nunique(), "(the paper: eight);", t72.groupby("neuron").size().to_dict())
print(f"pre-stimulus SD of the inverted whole-frame dF: median {t72['pre_sd/whole'].median():.2f}%")
for key in ["whole/a", "whole/b", "whole/a5sd", "whole/b5sd"]:
    roi = "whole"
    d = t72.assign(field=T.field(t72.voltage), n=t72[f"naps/{key}"],
                   lat=t72[f"aps/{key}"].apply(lambda a: a[0] if len(a) else np.nan), last=t72["last100/whole"])
    pf = d.groupby(["neuron", "field"]).agg(n=("n", "mean"), lat=("lat", "mean"), last=("last", "mean")).reset_index()
    exc = pf.groupby("neuron").n.max() > 0
    rho = {nr: stats.spearmanr(g.field, g.lat).statistic for nr, g in pf[pf.n > 0].groupby("neuron") if len(g) >= 3}
    blk = {}
    for nr, g in pf.groupby("neuron"):
        f = g[g.n > 0].sort_values("field")
        hi = g.sort_values("field").iloc[-1]
        blk[nr] = bool(len(f) and hi["last"] > 5 and hi["last"] > f.iloc[0]["last"])
    print(f"{key}: excited {int(exc.sum())}/{len(exc)}; rho {dict((k[-14:], round(v, 2)) for k, v in rho.items())}; "
          f"block {sum(blk.values())}/{len(blk)}; APs per neuron {d.groupby('neuron').n.sum().to_dict()}")
print("last-100-ms inverted dF at the highest field per neuron:",
      t72.assign(field=T.field(t72.voltage)).sort_values("field").groupby("neuron")["last100/whole"].last().round(1).to_dict())

**Result: not reproduced.** 001272 holds 3 neurons, where the paper used eight. The readings disagree on
excitation (3 of 3 neurons under reading a and both 5-SD readings, 1 of 3 under reading b at 5%), but under
every reading no neuron meets the block test at its highest field (0 of 3; inverted last-100-ms dF 0.7%,
3.3% and 1.6%), so the claim fails under all readings for the neurons deposited. Latency falls with field in
1 of 2 neurons with three fields with APs under reading (a), and in 2 of 3 under the 5-SD readings. Whether
the five neurons not deposited would change this cannot be known.

## 13. figs2-ttx-blocks-depolarization (DANDI:001271)

> 1 uM tetrodotoxin abolished both the APs and the sustained depolarization caused by intense unmodulated
> 2-kHz sine waves.

Per neuron, the session in normal solution and the session in TTX at the same field (the highest field
recorded in both; the latest normal session and the first TTX session at it). Reproduced when, in every TTX
neuron, APs present before are absent after, and the last-100-ms mean dF is above 5% before and below 5% after,
while a vehicle neuron keeps its response.

In [ ]:
d71 = t71.assign(field=T.field(t71.voltage))
rows = []
for nr, g in d71.groupby("neuron"):
    tt = g[g.ttx & ~g.sham]; vv = g[~g.ttx & ~g.sham]
    if not len(tt):
        if g.ttx.any():
            print(f"{nr}: tetrodotoxin sessions are shams only ({int((g.ttx & g.sham).sum())})")
        continue
    common = sorted(set(tt.field) & set(vv.field))
    f = common[-1]
    b = vv[vv.field == f].iloc[-1]; a = tt[tt.field == f].iloc[0]
    rows.append(dict(neuron=nr, field=f, minutes_apart=round((pd.Timestamp(a.start) - pd.Timestamp(b.start)).total_seconds() / 60, 1),
                     aps_before_a=b["naps/whole/a"], aps_after_a=a["naps/whole/a"],
                     aps_before_b=b["naps/whole/b"], aps_after_b=a["naps/whole/b"],
                     last100_before=round(b["last100/whole"], 1), last100_after=round(a["last100/whole"], 1),
                     max_last100_vehicle=round(vv["last100/whole"].max(), 1), max_last100_ttx=round(tt["last100/whole"].max(), 1),
                     ttx_sessions=len(tt), ttx_sessions_with_aps_a=int((tt["naps/whole/a"] > 0).sum()),
                     ttx_sessions_with_aps_b=int((tt["naps/whole/b"] > 0).sum())))
s2 = pd.DataFrame(rows)
print(s2.to_string(index=False))
print("TTX neurons where APs before and none after: reading a", int(((s2.aps_before_a > 0) & (s2.aps_after_a == 0)).sum()),
      ", reading b", int(((s2.aps_before_b > 0) & (s2.aps_after_b == 0)).sum()), "of", len(s2))
print("TTX neurons with last-100-ms dF > 5% before and < 5% after:",
      int(((s2.last100_before > 5) & (s2.last100_after < 5)).sum()), "of", len(s2))
veh = d71[~d71.neuron.isin(s2.neuron)]
print("neurons never in TTX:", veh.neuron.nunique(), "; sessions with last-100-ms dF > 5% among them:",
      int((veh["last100/whole"] > 5).sum()), "of", len(veh))
print("all TTX sessions, last-100-ms dF range:", round(d71[d71.ttx]["last100/whole"].min(), 1), "to",
      round(d71[d71.ttx]["last100/whole"].max(), 1))

**Result: not reproduced.** In all 6 neurons recorded in tetrodotoxin, APs present in normal solution at the
same field are absent in tetrodotoxin under reading (a) (5 of 6 under reading b, which finds rises of 5% in 15
of 17 tetrodotoxin sessions of one neuron). The last-100-ms dF in tetrodotoxin is at most 0.4% in every
session. But only 1 of the 6 neurons had a last-100-ms dF above 5% before tetrodotoxin (5.4%), and none of 171
sessions of the 14 neurons never put in tetrodotoxin did, so the block of sustained depolarization is shown in
one neuron, and the criterion, which asks it of every neuron, fails. The sessions compared are 3.6 to 11 min
apart, where the legend says about 5 min. The vehicle neuron of Figure S2 is not identified in the files; the
neurons never put in tetrodotoxin stand in for it.

## 14. Blocked claims, and the strobe images as a variant

fig3b-low-conductivity-halves-threshold. No session of 001032, 001033 or 001035 is described as made in
the sodium gluconate solution; every description gives the NaCl solution. 001030 is in sodium gluconate but
holds only 2-s ramped stimuli. The NaCl to NaGlu comparison cannot be found in the deposited files. Nor do the thresholds show it: the cell
after the Figure 4 listing splits each neuron's sessions into ascending voltage series and finds no series
whose threshold drops to 0.6 or less of the series before and after it, as a reversible switch to sodium
gluconate would.

Figure 4 (001273). The files hold raw strobe stacks only. The paper's ROIs over the most de- and
hyperpolarized regions are not deposited, and no file says which cell is elongated or random, nor (in most
files) which way the field was applied. The session counts are printed below.

Figures 5, 6 and S3 (001274). No ROIs (the claims file's O1 blocks these claims as
metadata_insufficient when the dataset holds none), no field values in V/cm, and cells C7 and C8 cannot be
identified (O4). As the claims file asks, ROIs rebuilt from the RATIO image of the Methods are shown as a
variant only, for the pulse measure of Figure 6B, on the nine subjects that have a pulse and 100-Hz sine
waves. The field is taken as the paper's 257 V/cm for a 10-V pulse (Figure 6A legend), scaled by the
generator voltage (one subject had 15 V), which the files do not state. The frame times assume that the
recording runs forward from 1 us before the pulse; all nine pulses used are described as forward, and the
difference of the two ROIs, averaged over the nine, is above half its maximum from frame 25 to 62 (printed
below), close to frames 20 to 59, the 2-us pulse at 50-ns steps.
These ROIs are placed at the extremes of the same RATIO image they are then measured on, which biases the
pulse value upwards.

In [ ]:
d73 = pd.DataFrame([dict(subject=a["path"].split("/")[0],
                         direction=("up-bottom" if "Up" in a["description"] else "right-left" if "Right and Left" in a["description"] else "not specified"),
                         volts=re.search(r"pulse at ([\d,]+)V", a["description"]).group(1)) for a in assets["001273"]])
print("001273:", len(d73), "files,", d73.subject.nunique(), "subjects; field direction in descriptions:",
      d73.direction.value_counts().to_dict())
print(d73.groupby(["subject", "direction"]).size().unstack(fill_value=0).to_string())

In [ ]:
# Was any neuron of 001032 to 001035 recorded in sodium gluconate although described as in NaCl? Figure 3B's
# switch "reversibly" lowered thresholds about 2-fold. Per neuron and unmodulated carrier, the sessions are split
# into ascending series (a new series starts when the voltage drops), and each series' 1-AP threshold is the
# lowest voltage with an AP (reading a). A reversible drop is a series with a threshold at most 0.6 times those
# of the series before and after it.
rows = []
for (n, c), g in fast[~fast.modulated & fast.carrier.notna()].sort_values("start").groupby(["neuron", "carrier"]):
    series = np.cumsum(np.r_[0, np.diff(g.voltage.values) < 0])
    th_s = [g[(series == k) & (g["naps/whole/a"] > 0)].voltage.min() for k in range(series.max() + 1)]
    th_s = [x for x in th_s if np.isfinite(x)]
    if len(th_s) >= 2:
        drop = any(th_s[k] <= 0.6 * min(th_s[k - 1], th_s[k + 1]) for k in range(1, len(th_s) - 1))
        rows.append(dict(neuron=n, carrier=c, series=len(th_s), thresholds_Vpp=th_s,
                         lowest_over_highest=round(min(th_s) / max(th_s), 2), reversible_drop=drop))
gl = pd.DataFrame(rows)
print(f"neuron-carrier pairs with two or more series with APs: {len(gl)}; with a reversible drop to 0.6 or less: "
      f"{int(gl.reversible_drop.sum())}; lowest over highest series threshold, median {gl.lowest_over_highest.median():.2f}, "
      f"at most 0.6 in {int((gl.lowest_over_highest <= 0.6).sum())}")
print(gl.to_string(index=False))

In [ ]:
pulses = strobe[strobe.subject.isin(cand) & (strobe.kind == "pulse")].sort_values("start").groupby("subject").first()
desc74 = {a["path"]: a["description"] for a in assets["001274"]}
ptr = {}
def pulse_variant(row):
    stack = T.load_strobe(row.url)
    t = T.frame_times_pulse(stack.shape[0])
    ratio, control = T.ratio_image(stack, t)
    roi1, roi2, cell = T.rebuilt_rois(ratio, control)
    f = {k: stack[:, m].mean(1).astype(float) for k, m in (("roi1", roi1), ("roi2", roi2))}
    out = dict(frames=stack.shape[0], volts=row.volts,
               direction="forward" if "forward manner" in desc74[row.path] else "backward" if "backward manner" in desc74[row.path] else "not stated")
    ptr[row.name] = dict(frame_mean=stack.reshape(stack.shape[0], -1).mean(1), t=t,
                         diff=100 * (f["roi2"] / f["roi2"][t < 0].mean() - f["roi1"] / f["roi1"][t < 0].mean()))
    for k, x in f.items():
        f0 = x[t < 0].mean()
        dF = 100 * (x - f0) / f0
        out[f"{k} dF 1-2 us"] = dF[(t >= 1.0) & (t <= 2.0)].mean()
    out["ratio min"], out["ratio max"] = np.nanmin(np.where(cell, ratio, np.nan)), np.nanmax(np.where(cell, ratio, np.nan))
    return out
with ThreadPoolExecutor(8) as ex:
    pv = pd.DataFrame(list(ex.map(pulse_variant, [r for _, r in pulses.iterrows()])), index=pulses.index)
pv["field kV/cm"] = 0.257 * pv.volts / 10      # 257 V/cm at the 10-V pulse (Figure 6A legend), scaled by the voltage
for k in ("roi1", "roi2"):
    pv[f"{k} %/kV/cm"] = pv[f"{k} dF 1-2 us"].abs() / pv["field kV/cm"]
print(pv.round(2).to_string())
s = pd.concat([pv["roi1 %/kV/cm"], pv["roi2 %/kV/cm"]])
print(f"variant only: mean |dF| per kV/cm over the {len(s)} rebuilt ROIs {s.mean():.1f} +- {s.sem():.1f}, range "
      f"{s.min():.1f} to {s.max():.1f} (paper: 28 +- 2.5, range 15 to 47)")
# frame timing: the pulse is taken to start at frame 20 (50-ns steps from -1 us)
fig, axs = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
for sub, d in ptr.items():
    axs[0].plot(d["frame_mean"] / d["frame_mean"][1:20].mean(), lw=0.7)
    axs[1].plot(d["diff"], lw=0.7)
for ax in axs:
    ax.axvline(20, color="k", ls=":")
axs[0].set_ylabel("whole-frame mean / pre-pulse"); axs[1].set_ylabel("ROI 2 - ROI 1 dF (%)"); axs[1].set_xlabel("frame")
plt.tight_layout(); plt.show()
md = np.mean([d["diff"] for d in ptr.values()], axis=0)
sm = np.convolve(md, np.ones(5) / 5, mode="same")
half = np.where(sm[1:] > sm[1:].max() / 2)[0] + 1
print(f"ROI 2 - ROI 1 dF averaged over the {len(ptr)} subjects (5-frame running mean): frames 1 to 19 {md[1:20].mean():.1f}%, "
      f"20 to 39 {md[20:40].mean():.1f}%, 40 to 59 {md[40:60].mean():.1f}%, 60 to 79 {md[60:80].mean():.1f}%; above half its maximum "
      f"from frame {half.min()} to {half.max()} (the assumed pulse is frames 20 to 59)")
print("frames whose whole-frame mean is below 0.9 of the pre-pulse mean (flash dropouts), frame 0 left out:",
      {s: (np.where(d["frame_mean"][1:] / d["frame_mean"][1:20].mean() < 0.9)[0] + 1).tolist() for s, d in ptr.items()})


The same rebuilt ROIs on the 100-Hz sine wave sessions of these subjects (Figure 6C), averaged over sessions,
LOESS-subtracted at each span of the claims file's O2, and regressed frame by frame on the signed field (O3
reading a). The files hold neither the waveform nor the field, so the peak field is taken as the paper's 136
V/cm (Figure 5 legend) and the carrier phase is fitted to ROI 1; the sign of ROI 2 then says whether it
oscillates in antiphase. Reading (b) of O3 (half the peak-to-peak per cycle) is dominated by the noise of
single frames and is not shown. Fitting the phase on ROI 1 forces its slope to be positive and inflates its
p-value; ROI 2's sign is the informative one. A variant only.

In [ ]:
def sine_variant(subject):
    row = pulses.loc[subject]
    stack = T.load_strobe(row.url)
    t = T.frame_times_pulse(stack.shape[0])
    ratio, control = T.ratio_image(stack, t)
    roi1, roi2, _ = T.rebuilt_rois(ratio, control)
    files = strobe[(strobe.subject == subject) & (strobe.kind == "sine 100 Hz")]
    tr = [T.load_strobe(u, {"roi1": roi1, "roi2": roi2}) for u in files.url]
    out = []
    mean = {k: np.mean([x[k] for x in tr], axis=0) for k in ("roi1", "roi2")}
    dF = {k: T.sine_dF(v) for k, v in mean.items()}
    for span in T.LOESS_SPANS:
        sub = {k: v - T.loess(v, span) for k, v in dF.items()}
        phase = T.fit_phase(sub["roi1"], 136.0)
        fld = T.signed_field(len(sub["roi1"]), 136.0, phase)
        for k in ("roi1", "roi2"):
            slope, se, p = T.frame_regression(sub[k], fld)
            out.append(dict(subject=subject, span=span, roi=k, sessions=len(tr), slope=1000 * slope,
                            se=1000 * se, p=p))
    return out
with ThreadPoolExecutor(9) as ex:
    sv = pd.DataFrame([r for rows in ex.map(sine_variant, cand) for r in rows])
sv["sens"] = sv.slope.abs()
print(sv[sv.span == 0.75].round(4).to_string(index=False))
for span, g in sv.groupby("span"):
    pul = pd.concat([pv["roi1 %/kV/cm"].rename("roi1"), pv["roi2 %/kV/cm"].rename("roi2")], axis=1).stack()
    sens = g.set_index(["subject", "roi"]).sens
    ratio = (sens / pul.reindex(sens.index)).dropna()
    print(f"span {span}: mean sine sensitivity {sens.mean():.1f} +- {sens.sem():.1f} %/kV/cm (range {sens.min():.1f} to "
          f"{sens.max():.1f}); p < 0.0001 in {int((g.p < 1e-4).sum())} of {len(g)} ROIs; ROI 2 slope negative in "
          f"{int((g[g.roi == 'roi2'].slope < 0).sum())} of {g.subject.nunique()}; sine below pulse in {int((ratio < 1).sum())} of "
          f"{len(ratio)} ROIs; mean sine over mean pulse {sens.mean() / pul.mean():.2f}")

The rebuilt ROIs give a pulse value of 19.5 +- 2.4 %/kV/cm over 18 ROIs and a sine value of about 5.5 +- 0.8,
28% of the pulse value, below it in 17 of 18 ROIs, with ROI 2 in antiphase in 7 of 9 subjects. These are
variants on ROIs and a field the paper does not give, and decide no outcome.

## 15. Summary

| Claim | Outcome |
|---|---|
| fig1-pure-sine-excites-all | not reproduced (2 kHz 96%, 20 kHz 88% of neurons; 90% asked at each) |
| fig1-onset-delay | not reproduced (latency falls with field; median 64 ms, not 90 to 220) |
| fig1-max-rate-ceiling | not reproduced, resting on an assumption (39.1 Hz; reading b 42.9 Hz) |
| fig1-excitation-block | not reproduced (3 of 42) |
| fig2d-20khz-twofold-threshold | not reproduced (p < 0.0001, ratios 2.27 to 2.82) |
| fig2d-modulated-thresholds-higher | partially reproduced (2 of 3 pairs) |
| threshold-2khz-unmodulated | not reproduced (75.6 V/cm against 57.7) |
| fig2-modulated-locks-to-beats | not reproduced (locking holds; one session ends depolarized) |
| fig2c-pure-more-aps | reproduced |
| fig3b-low-conductivity-halves-threshold | blocked: metadata insufficient |
| fig3c-sustained-firing-2s | not reproduced (5 or 1 peaks after 250 ms at the highest field); reproduced proposed, as the peaks are at the noise rate |
| figs1-di8-anepps-control | not reproduced (3 of 8 neurons deposited) |
| figs2-ttx-blocks-depolarization | not reproduced (depolarization above 5% before TTX in 1 of 6) |
| fig4 (four claims) | blocked: metadata insufficient |
| fig5, figs3, fig6 (eight claims) | blocked: metadata insufficient |
| threshold-depolarization-estimate | blocked: metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's. The soma ROI is the whole 16 x 16 frame (assumption; the
bright half is a variant). APs are detected by the two readings of the claims file with an 8-ms merge; reading
(a) also asks a 5% prominence and a 1-ms half-width (assumption, section 3). The field is 36 V/cm times half the
generator's peak-to-peak voltage (assumption; the whole of it is a variant), with no frequency calibration,
which the dandisets say is needed and do not give. Neurons from 001032, 001033 and 001035 are pooled, as the
paper does not say which neurons form which figure.

What was found about the dandisets. 001274 and 001273 hold raw strobe stacks only: no ROIs, no field
values, no frame times, no cell shapes or groups, and no identifier for the paper's cells C1 to C9.
001034 is a second deposit of 001032. No 250-ms session is labelled as sodium gluconate, so Figure 3B cannot
be found. 001272 holds 3 of the 8 Di-8-ANEPPS neurons. The descriptions say tetrodotoxin "caused sustained
depolarization", where the paper and the data show none in tetrodotoxin. The voltages are uncalibrated, with
calibration factors "to be reported separately". The images are 16 x 16 pixels where the paper gives 64 x 64,
and the first frame of every recording is an artifact. 001030 makes one subject per session.

In [ ]:
print(f"notebook ran in {(time.time() - t_start) / 60:.1f} min")